In [1]:
import os
import warnings
os.chdir("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import BallTree
from scipy.stats import mannwhitneyu
from xgboost import XGBRegressor

In [2]:
df_listings = pd.read_csv('data/combined_csvs/listings_property_vals.csv')
df_listings = df_listings[~pd.isna(df_listings["gross_yield"])]


## Feature Engineering

In [3]:
# df_listings["zip3"] = df_listings["zipcode"].apply(lambda x: str(x)[0:3])

city_centers = {
    'raleigh': (35.7796, -78.6382),
    'charlotte': (35.2271, -80.8431),
    'durham': (35.9940, -78.8986),
    'asheville': (35.5951, -82.5515),
    'wilmington': (34.2104, -77.9447),
    'carolina beach': (34.0352, -77.8936),
    'myrtle beach': (33.6891, -78.8867),
    'pigeon forge': (35.7884, -83.5543),
    'gatlinburg': (35.7143, -83.5102),
    'williamsburg': (37.2695, -76.7084)
}

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculates the distance in miles between two lat/lon points."""
    # Convert decimal degrees to radians 
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula 
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a)) 
    
    # 3956 is the radius of the Earth in miles. (Use 6371 for kilometers)
    miles = 3956 * c
    return miles

In [4]:
# Distance to city centers
df_listings['city_lower'] = df_listings['city'].str.lower().str.strip()
df_listings['center_lat'] = df_listings['city_lower'].map(lambda x: city_centers.get(x, (np.nan, np.nan))[0])
df_listings['center_lon'] = df_listings['city_lower'].map(lambda x: city_centers.get(x, (np.nan, np.nan))[1])
df_listings['dist_to_center_miles'] = haversine_distance(
    df_listings['latitude'],
    df_listings['longitude'],
    df_listings['center_lat'],
    df_listings['center_lon']
)
df_listings = df_listings.drop(columns=['city_lower', 'center_lat', 'center_lon'])

In [5]:
# ratio of cleaning fee to average rate
df_listings['cleaning_fee_ratio'] = df_listings['cleaning_fee'] / df_listings['ttm_avg_rate']

In [6]:
# one hot encode amenities
amenities_encoded = df_listings['amenities'].str.get_dummies(sep=',')
df_listings = df_listings.join(amenities_encoded)

In [7]:
def should_keep_amenity(col, target, min_freq=0.05, max_freq=0.95, alpha=0.01):
    freq = col.mean()
    
    # 1. If it falls in the healthy variance zone, keep it automatically
    if min_freq <= freq <= max_freq:
        return True
    
    # 2. If it's too ubiquitous (e.g., > 98%), drop it. It offers no contrast.
    if freq > 0.98:
        return False
        
    # 3. If it's rare, check if the price distribution is significantly different
    # when the amenity is present vs. when it's absent
    price_with = target[col == 1]
    price_without = target[col == 0]
    
    # Ensure we have at least a few samples to test
    if len(price_with) < 5:
        return False
        
    # Non-parametric test to see if 'rare' actually moves the needle on price
    stat, p_val = mannwhitneyu(price_with, price_without, alternative='two-sided')
    
    # Keep if statistically significant
    return p_val < alpha

# Apply the conditional filter
keep_columns = [
    col for col in amenities_encoded.columns 
    if should_keep_amenity(amenities_encoded[col], df_listings['ttm_revenue']) # replacing target with your price column
]

amenities_filtered = amenities_encoded[keep_columns]

X = amenities_filtered
significant_features = list(amenities_filtered.columns)

In [8]:
import numpy as np
import pandas as pd
import warnings
from sklearn.neighbors import BallTree
from scipy.stats import spearmanr

def find_best_k_for_competitors(df, target_col='ttm_revenue', k_candidates=range(100, 250)):
    """
    Dynamically finds the best k by testing which neighborhood size 
    correlates most strongly with the target variable.
    """
    print("Testing k candidates...")
    
    # 1. Prepare locations and tree
    locations = np.deg2rad(df[['latitude', 'longitude']].values)
    tree = BallTree(locations, metric='haversine')
    ratings = df['rating_overall'].values
    
    # We use the log of revenue for a more stable correlation test
    target_log = np.log1p(df[target_col].values)
    global_mean_rating = df['rating_overall'].mean()
    
    best_k = None
    best_corr = 0
    best_feature = None
    
    # 2. Iterate through candidate k values
    for k in k_candidates:
        # Query tree
        distances, indices = tree.query(locations, k=k+1)
        comp_indices = indices[:, 1:]
        comp_ratings = ratings[comp_indices]
        
        # Calculate average ratings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            avg_comp_ratings = np.nanmean(comp_ratings, axis=1)
            
        # Fill NaNs with global mean for the correlation test
        avg_comp_ratings = np.nan_to_num(avg_comp_ratings, nan=global_mean_rating)
        
        # 3. Calculate Spearman correlation (robust to outliers)
        corr, _ = spearmanr(avg_comp_ratings, target_log)
        corr = abs(corr)  # We care about the strength of the signal
        
        print(f"k={k:<2} | Spearman Correlation = {corr:.4f}")
        
        # 4. Track the best performing k
        if corr > best_corr:
            best_corr = corr
            best_k = k
            best_feature = avg_comp_ratings
            
    print(f"\nSelected best k: {best_k} (Correlation: {best_corr:.4f})")
    return best_k, best_feature

# Run the search
best_k, optimized_rating_feature = find_best_k_for_competitors(
    df_listings, 
    target_col='ttm_revenue',
)

# Create the new column name
new_feature_name = f'avg_comp_rating'

# Assign the winning feature to the dataframe dynamically
df_listings[new_feature_name] = optimized_rating_feature
    
print(f"Successfully added '{new_feature_name}' to X_col predictors.")

Testing k candidates...
k=100 | Spearman Correlation = 0.1066
k=101 | Spearman Correlation = 0.1080
k=102 | Spearman Correlation = 0.1083
k=103 | Spearman Correlation = 0.1091
k=104 | Spearman Correlation = 0.1107
k=105 | Spearman Correlation = 0.1106
k=106 | Spearman Correlation = 0.1122
k=107 | Spearman Correlation = 0.1106
k=108 | Spearman Correlation = 0.1099
k=109 | Spearman Correlation = 0.1091
k=110 | Spearman Correlation = 0.1090
k=111 | Spearman Correlation = 0.1079
k=112 | Spearman Correlation = 0.1080
k=113 | Spearman Correlation = 0.1096
k=114 | Spearman Correlation = 0.1088
k=115 | Spearman Correlation = 0.1106
k=116 | Spearman Correlation = 0.1106
k=117 | Spearman Correlation = 0.1114
k=118 | Spearman Correlation = 0.1115
k=119 | Spearman Correlation = 0.1112
k=120 | Spearman Correlation = 0.1121
k=121 | Spearman Correlation = 0.1124
k=122 | Spearman Correlation = 0.1117
k=123 | Spearman Correlation = 0.1108
k=124 | Spearman Correlation = 0.1103
k=125 | Spearman Correlati

## Define Predictors

In [ ]:
X_col = [
    "property_val",
    "room_type",
    "bedrooms",
    "baths",
    "ttm_blocked_days",
    "SizeRank",
    "city",
    "dist_to_center_miles",
    "instant_book",
    "photos_count",
    "professional_management",
    "cleaning_fee_ratio",
    "avg_comp_rating",
    ] + significant_features 
Y_col = ["ttm_revenue"]

## Check Multicolinearity

In [10]:
def calculate_gvif(df, features):
    """
    Automatically preprocesses categorical data and calculates GVIF.
    
    Parameters:
    df: Raw pandas DataFrame (can contain text/objects).
    features: List of column names to include in the model.
              e.g., ['property_val', 'beds', 'room_type', 'zipcode']
    """
    # 1. Isolate only the requested features and drop missing values
    # (GVIF requires a complete matrix without NaNs)
    df_subset = df[features].dropna()
    
    # 2. Identify numeric vs categorical columns
    numeric_cols = df_subset.select_dtypes(include=['number', 'bool']).columns.tolist()
    cat_cols = df_subset.select_dtypes(exclude=['number', 'bool']).columns.tolist()
    
    # 3. One-hot encode categorical variables automatically
    # dtype=float prevents pandas boolean issues in numpy matrices
    df_encoded = pd.get_dummies(df_subset, columns=cat_cols, drop_first=True, dtype=float)
    
    # 4. Build the column mapping internally
    feature_groups = {}
    for col in numeric_cols:
        feature_groups[col] = [col]
        
    for col in cat_cols:
        # Find all dummy columns generated for this original categorical column
        dummy_cols = [c for c in df_encoded.columns if c.startswith(f"{col}_")]
        if dummy_cols: 
            feature_groups[col] = dummy_cols

    # 5. Calculate GVIF
    corr_matrix = df_encoded.corr().values
    det_all = np.linalg.det(corr_matrix)
    
    # Safety check for perfect multicollinearity
    if np.isclose(det_all, 0):
        raise ValueError("Perfect multicollinearity detected. Determinant is 0. Check your features for duplicates or exact linear combinations.")
    
    results = []
    
    for feature_name, columns in feature_groups.items():
        # Get integer indices of the columns for this feature block
        col_indices = [df_encoded.columns.get_loc(col) for col in columns]
        
        # Get integer indices of all OTHER columns
        other_indices = [i for i in range(df_encoded.shape[1]) if i not in col_indices]
        
        # Determinants
        if len(col_indices) > 1:
            det_feature = np.linalg.det(df_encoded.iloc[:, col_indices].corr().values)
        else:
            det_feature = 1.0 
            
        det_other = np.linalg.det(df_encoded.iloc[:, other_indices].corr().values)
        
        # Calculate GVIF
        gvif = (det_feature * det_other) / det_all
        
        # Calculate Degrees of Freedom
        df_val = len(col_indices)
        
        # Calculate Adjusted GVIF
        gvif_adjusted = gvif ** (1 / (2 * df_val))
        
        results.append({
            'Feature': feature_name,
            'GVIF': gvif,
            'df': df_val,
            'GVIF^(1/2df)': gvif_adjusted
        })
        
    # Return results sorted by the adjusted GVIF
    return pd.DataFrame(results).sort_values(by='GVIF^(1/2df)', ascending=False)

In [11]:
calculate_gvif(df_listings, X_col)

/opt/anaconda3/envs/iaa/lib/python3.14/site-packages/numpy/linalg/_linalg.py:2406: RuntimeWarning: invalid value encountered in det
  r = _umath_linalg.det(a, signature=signature)


,Feature,GVIF,df,GVIF^(1/2df)
0,property_val,NaN,1,NaN
1,bedrooms,NaN,1,NaN
2,baths,NaN,1,NaN
3,ttm_blocked_days,NaN,1,NaN
4,SizeRank,NaN,1,NaN
...,...,...,...,...
104,Wine glasses,NaN,1,NaN
105,room_type,NaN,2,NaN
106,city,NaN,9,NaN
107,instant_book,NaN,1,NaN


## Preprocessing

In [ ]:
# filter out rows with missing target value
df_listings['instant_book'] = df_listings['instant_book'].astype(str).str.upper().map({'TRUE': 1, 'FALSE': 0}).fillna(0).astype(int)
df_listings['professional_management'] = df_listings['professional_management'].astype(str).str.upper().map({'TRUE': 1, 'FALSE': 0}).fillna(0).astype(int)

# cast bools to 0 and 1
bool_cols = df_listings.select_dtypes(include='bool').columns
df_listings[bool_cols] = df_listings[bool_cols].astype(int)

y = df_listings[Y_col]
X = df_listings[X_col]

# set str columns as categories
categorical_cols = ["room_type", "city"]
X[categorical_cols] = X[categorical_cols].astype("category")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Model Training

In [13]:
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method="hist",  # Fast histogram optimized training
    enable_categorical=True,
    early_stopping_rounds=50,
    n_jobs=-1,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,  # Prints log every 50 trees
)

preds = model.predict(X_test)
rmse = root_mean_squared_error(y_test, preds)
mae = mean_absolute_error(y_test, preds)
mape = mean_absolute_percentage_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f"\nModel Performance:")
print(f"Best Iteration (Tree Count): {model.best_iteration}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MAPE {mape:.4f}")
print(f"R² Score: {r2:.4f}")

importance = pd.DataFrame(
    {"Feature": X.columns, "Importance": model.feature_importances_}
).sort_values(by="Importance", ascending=False)
print("\nFeature Importance:")
print(importance.to_string(index=False))

[0]	validation_0-rmse:28339.47384
[50]	validation_0-rmse:21941.70806
[100]	validation_0-rmse:20947.98987
[150]	validation_0-rmse:20490.98138
[200]	validation_0-rmse:20385.11336
[250]	validation_0-rmse:20328.81128
[300]	validation_0-rmse:20360.01569
[313]	validation_0-rmse:20345.64993

Model Performance:
Best Iteration (Tree Count): 263
RMSE: 20295.7754
MAE: 13744.0488
MAPE 0.9494
R² Score: 0.4982

Feature Importance:
                              Feature  Importance
                 Lock on bedroom door    0.051024
                         property_val    0.033139
                            room_type    0.029711
                           High chair    0.028945
                              Freezer    0.027962
             Pack ’n play/Travel crib    0.020297
                              zipcode    0.019718
                               Washer    0.017347
                    Fire extinguisher    0.017318
                     ttm_blocked_days    0.017206
                         inst

In [14]:
len(df_listings)

2670

In [15]:
from sklearn.inspection import permutation_importance

# Assuming xgb_model is already fit
result = permutation_importance(
    model, X_test, y_test, n_repeats=10, random_state=42, scoring='neg_root_mean_squared_error'
)

# Organize into a DataFrame
perm_importances = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': result.importances_mean
}).sort_values(by='Importance', ascending=False)

In [16]:
# pd.set_option('display.max_rows', None)
# perm_importances

In [17]:
X_col_new = list(perm_importances[perm_importances["Importance"] >= 40]["Feature"])
X_col_new

['ttm_blocked_days',
 'property_val',
 'zipcode',
 'Lock on bedroom door',
 'photos_count',
 'cleaning_fee_ratio',
 'avg_comp_rating',
 'baths',
 'instant_book',
 'bedrooms',
 'Cable TV',
 'Hot tub',
 'dist_to_center_miles',
 'Ceiling fan',
 'High chair',
 'room_type',
 'Freezer',
 'Pets allowed',
 'Pool table',
 'Fire extinguisher',
 'Shower gel',
 'Indoor fireplace',
 'Iron',
 'Dining table',
 'Exterior security cameras on property',
 'Baking sheet',
 'Beach access',
 'Sun loungers',
 'Coffee',
 'Single level home',
 'Hair dryer',
 'Wine glasses',
 'professional_management']

In [18]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

# 1. Keep target on original scale
X = df_listings[X_col]
y = df_listings[Y_col]

categorical_cols = ["room_type", "city"]
X[categorical_cols] = X[categorical_cols].astype("category")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Fixed custom evaluation metric for early stopping evaluation
def xgboost_neg_r2(y_pred, dtrain):
    y_true = dtrain
    return -r2_score(y_true, y_pred)

# 3. Model setup with explicit maximization control
model = XGBRegressor(
    n_estimators=600,
    learning_rate=0.03,      
    max_depth=6,             
    subsample=0.8,
    colsample_bytree=0.7,    
    reg_lambda=50,           
    reg_alpha=10,            
    eval_metric=xgboost_neg_r2,
    early_stopping_rounds=50,
    # We want to minimize the negative R² score (meaning maximize actual R²)
    maximize=False,           
    random_state=42,
    tree_method="hist",
    enable_categorical=True,
    n_jobs=-1,
)

# 4. Fit model
model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)

# 5. Predict and evaluate
preds = model.predict(X_test)
preds = np.clip(preds, a_min=0, a_max=None)

rmse = root_mean_squared_error(y_test, preds)
mae = mean_absolute_error(y_test, preds)
mape = mean_absolute_percentage_error(y_test, preds)
r2 = r2_score(y_test, preds)
# Flatten y_test to a 1D numpy array so its shape perfectly matches preds
y_test_array = y_test.values.flatten()
# Calculate WMAPE
wmape = np.sum(np.abs(y_test_array - preds)) / np.sum(y_test_array)

print(f"WMAPE: {wmape:.4f} (or {wmape * 100:.2f}%)")

print(f"\nModel Performance (Optimized Directly on R²):")
print(f"Best Iteration (Tree Count): {model.best_iteration}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MAPE: {mape:.4f}")
print(f"WMAPE: {wmape:.4f} (or {wmape * 100:.2f}%)")
print(f"R² Score: {r2:.4f}")

importance = pd.DataFrame(
    {"Feature": X.columns, "Importance": model.feature_importances_}
).sort_values(by="Importance", ascending=False)

print("\nTop 20 Feature Importance:")
print(importance.head(20).to_string(index=False))

[0]	validation_0-rmse:28560.18724	validation_0-xgboost_neg_r2:6909.34863
[50]	validation_0-rmse:23657.63538	validation_0-xgboost_neg_r2:5.78145


/opt/anaconda3/envs/iaa/lib/python3.14/site-packages/xgboost/callback.py:385: UserWarning: [17:38:44] WARNING: /Users/ec2-user/croot/xgboost-split_1783526132119/work/src/learner.cc:793: 
Parameters: { "maximize" } are not used.

  self.starting_round = model.num_boosted_rounds()


[100]	validation_0-rmse:21739.80782	validation_0-xgboost_neg_r2:1.82084
[150]	validation_0-rmse:20741.67666	validation_0-xgboost_neg_r2:0.91848
[200]	validation_0-rmse:20267.04950	validation_0-xgboost_neg_r2:0.60602
[250]	validation_0-rmse:20022.50448	validation_0-xgboost_neg_r2:0.45955
[300]	validation_0-rmse:19818.88810	validation_0-xgboost_neg_r2:0.35930
[350]	validation_0-rmse:19686.61491	validation_0-xgboost_neg_r2:0.29598
[400]	validation_0-rmse:19615.14827	validation_0-xgboost_neg_r2:0.26427
[450]	validation_0-rmse:19540.02185	validation_0-xgboost_neg_r2:0.22762
[500]	validation_0-rmse:19502.37918	validation_0-xgboost_neg_r2:0.20456
[550]	validation_0-rmse:19477.22533	validation_0-xgboost_neg_r2:0.18805
[599]	validation_0-rmse:19451.20579	validation_0-xgboost_neg_r2:0.16955
WMAPE: 0.3959 (or 39.59%)

Model Performance (Optimized Directly on R²):
Best Iteration (Tree Count): 599
RMSE: 19450.8574
MAE: 12633.1406
MAPE: 0.8524
WMAPE: 0.3959 (or 39.59%)
R² Score: 0.5391

Top 20 Featu

In [19]:
import optuna
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

def objective(trial):
    # 1. Define the hyperparameter search space
    param = {
        "n_estimators": 1500,  # Keep this high, early stopping will catch it
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "subsample": trial.suggest_float("subsample", 0.5, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.9),
        
        # Regularization to handle those noisy amenities
        "reg_lambda": trial.suggest_float("reg_lambda", 1, 100, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1, 100, log=True),
        
        # Minimum sum of instance weight needed in a child (prevents overfitting specific property types)
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        
        # Static parameters
        "eval_metric": "rmse",
        "early_stopping_rounds": 75,
        "random_state": 42,
        "tree_method": "hist",
        "enable_categorical": True,
        "n_jobs": -1,
    }

    # 2. Initialize the model with the suggested parameters
    model = XGBRegressor(**param)

    # 3. Fit the model
    # We turn verbose=False so it doesn't print 1,000 lines per trial
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        verbose=False, 
    )

    # 4. Predict and evaluate
    preds = model.predict(X_test)
    preds = np.clip(preds, a_min=0, a_max=None)
    
    # We want Optuna to maximize the R² score
    r2 = r2_score(y_test, preds)
    return r2

# 5. Create a study object and specify the direction is 'maximize'
# We use a fixed seed in the sampler for reproducibility 
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)

print("Starting Optuna optimization...")
# Run 50 trials (you can increase this to 100+ if you have time/compute)
study.optimize(objective, n_trials=50)

# 6. Output the final champion model parameters
print("\n" + "="*40)
print(f"🏆 Best Trial R² Score: {study.best_value:.4f}")
print("="*40)
print("Best Hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

[I 2026-07-20 17:38:45,814] A new study created in memory with name: no-name-dd9870da-3f64-4d7e-a843-232cd7b7f563


Starting Optuna optimization...


[I 2026-07-20 17:38:47,179] Trial 0 finished with value: 0.5311018228530884 and parameters: {'learning_rate': 0.023688639503640783, 'max_depth': 8, 'subsample': 0.7927975767245621, 'colsample_bytree': 0.659195090518222, 'reg_lambda': 2.05133826308745, 'reg_alpha': 2.0511104188433973, 'min_child_weight': 1}. Best is trial 0 with value: 0.5311018228530884.
[I 2026-07-20 17:38:48,327] Trial 1 finished with value: 0.5390245914459229 and parameters: {'learning_rate': 0.07348118405270448, 'max_depth': 6, 'subsample': 0.7832290311184182, 'colsample_bytree': 0.31235069657748143, 'reg_lambda': 87.06020878304857, 'reg_alpha': 46.22589001020831, 'min_child_weight': 3}. Best is trial 1 with value: 0.5390245914459229.
[I 2026-07-20 17:38:49,706] Trial 2 finished with value: 0.540261447429657 and parameters: {'learning_rate': 0.015199348301309814, 'max_depth': 4, 'subsample': 0.621696897183815, 'colsample_bytree': 0.6148538589793427, 'reg_lambda': 7.309539835912911, 'reg_alpha': 3.8234752246751853, 


🏆 Best Trial R² Score: 0.5543
Best Hyperparameters:
    learning_rate: 0.016247164702430712
    max_depth: 5
    subsample: 0.6557287851247491
    colsample_bytree: 0.41499985518355187
    reg_lambda: 1.877085618973148
    reg_alpha: 68.13272470766717
    min_child_weight: 9
